# PaliGemma 2: Türkçe OCR, 224 vs 448 (v3)
Soru: çözünürlük artışı (224 → 448) Türkçe tabela/fiş okumada (ş ğ ı ç ö ü) fark yaratıyor mu?

**v2'den farklar**
1. **Hedef düzeltmesi:** fotoğrafta fiş 1 `GURME` (ü yok), fiş 5 `ISL.LTD.STI.` (Türkçe harfsiz). v2'deki `gürme` ve `ŞTİ` hedefleri fotoğrafa uymuyordu.
2. **Uydurma sütunu:** her fotoğrafın tam metni (`TAM_METIN`) yazıldı; çıktıdaki, fotoğrafta olmayan ve fotoğraftaki hiçbir sözcüğe benzemeyen (harf hatası olmayan) sözcükler sayılıyor. Kart numaraları gibi kişisel rakamlar bu kopyadan çıkarıldı (rakamlar zaten sayılmıyor).
3. Kutu/tabela hedefleri artık kodda.
4. `REPEAT` ve `kosu` sütunu. **Bulgu:** aynı oturumda 3 koşu birbirinin aynısı çıktı (110 hücrenin 110'unda aynı metin), yani greedy kod aynı oturumda deterministik. Oynama **farklı Colab oturumları arasında** var (v1, v2, v3 ayrı oturumlar); bunun için `compare_sessions.py` ve `results/` klasörüne bak. Bu yüzden `REPEAT = 1` yeterli.

Koşullar: 2 çözünürlük × 5 prompt × 11 foto, greedy, `max_new_tokens=100`.
Çalışma zamanı: GPU (T4). Önce HF'de `google/paligemma2-3b-mix-224` ve `-448` lisanslarını kabul et.


In [ ]:
!pip -q install -U transformers accelerate pillow-heif

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Fotoğrafları yükle
İki yoldan biri:
- Drive: `MyDrive/fotolar/` içinde duruyorsa aşağıdaki hücre bağlar.
- Elle: sol panel → Files → `fotolar/` aç → görselleri sürükle. Dosya adı = örnek adı (`tabela_01.jpg`, `fis_01.jpg`).
Aynı 11 fotoğrafı kullan (v1 ile karşılaştırma için).

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive bağlanmadı, elle yüklenen fotolar/ kullanılacak:", type(e).__name__)

In [ ]:
import os, re, glob, gc, torch, pandas as pd
from PIL import Image
from pillow_heif import register_heif_opener
register_heif_opener()  # iPhone HEIC dosyaları .jpg adıyla gelse de açılsın
import transformers; transformers.logging.set_verbosity_error()  # <image> uyarısı zararsız
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

FOTO_DIR = "/content/drive/MyDrive/fotolar" if os.path.isdir("/content/drive/MyDrive/fotolar") else "fotolar"
os.makedirs(FOTO_DIR, exist_ok=True)
paths = sorted(p for p in glob.glob(f"{FOTO_DIR}/*") if p.lower().endswith((".jpg",".jpeg",".png",".webp")))
print(FOTO_DIR, "→", len(paths), "görsel")

MODELS = {224: "google/paligemma2-3b-mix-224", 448: "google/paligemma2-3b-mix-448"}
# Ad = <dil kodu>_<sorunun dili>. İlk üçü v1'deki promptlar (ocr, tr→tr_tr, en→en_en), son ikisi YENİ çapraz hücreler.
PROMPTS = {
    "ocr":   "ocr",
    "en_en": "answer en What does the text in the image say?",
    "tr_tr": "answer tr Görselde ne yazıyor?",
    "tr_en": "answer tr What does the text in the image say?",   # YENİ: dil kodu tr, soru İngilizce
    "en_tr": "answer en Görselde ne yazıyor?",                    # YENİ: dil kodu en, soru Türkçe
}
# T4 bf16'yı donanımda desteklemez; çalışır ama yavaş olabilir. Hata verirse torch.float16 dene
# (Gemma 2'de fp16 taşma yapabilir, çıktı bozuksa bf16'ya dön).
DTYPE = torch.bfloat16
REPEAT = 3   # aynı kurulum kaç kez koşsun (oynama ölçümü için)

# Çalışmadan önce bozuk dosyaları bul
good, bad = [], []
for p in paths:
    try:
        Image.open(p).convert("RGB"); good.append(p)
    except Exception as e:
        bad.append((p, os.path.getsize(p), type(e).__name__))
print("açılan:", len(good), "| açılmayan:", bad)
paths = good  # bozukları atla; yukarıdaki listeye bakıp yeniden yükle

In [ ]:
rows = []
for res, mid in MODELS.items():
    model = PaliGemmaForConditionalGeneration.from_pretrained(mid, torch_dtype=DTYPE, device_map="auto").eval()
    proc = PaliGemmaProcessor.from_pretrained(mid)
    for kosu in range(1, REPEAT + 1):
        for p in paths:
            img = Image.open(p).convert("RGB")
            W, H = img.size
            for pname, prompt in PROMPTS.items():
                inp = proc(text=prompt, images=img, return_tensors="pt").to(DTYPE).to(model.device)
                n = inp["input_ids"].shape[-1]
                with torch.inference_mode():
                    out = model.generate(**inp, max_new_tokens=100, do_sample=False)
                text = proc.decode(out[0][n:], skip_special_tokens=True)
                rows.append(dict(kosu=kosu, foto=os.path.basename(p), cozunurluk=res, prompt=pname, cikti=text, W=W, H=H))
        print(res, "koşu", kosu, "tamam")
        pd.DataFrame(rows).to_csv("sonuclar_v3.csv", index=False)   # her koşudan sonra yedek
    del model, proc; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv("sonuclar_v3.csv", index=False)
print(len(df), "çıktı kaydedildi (beklenen:", len(paths)*len(MODELS)*len(PROMPTS)*REPEAT, ")")
try:
    from google.colab import files; files.download("sonuclar_v3.csv")   # Drive'a da kopyala: !cp sonuclar_v3.csv /content/drive/MyDrive/
except Exception: pass
df.head(10)


## Harf yüksekliği (piksel)
Her fotoğraf aşağıda orijinal piksel eksenleriyle çiziliyor. **Hedef satırın** (fişte işletme adı, kutuda uyarı cümlesi) büyük harfinin yüksekliğini (örn. `E` harfinin üstünden altına) eksenlerden oku, `HARF_PX` sözlüğüne yaz. Kabaca yeter (±%10).
Model görseli kare `res×res`'e sıkıştırır, o yüzden modelin gördüğü harf boyu = `harf_px × res / H`. Patch 14 px, yani `÷14` = harfin kaç patch yüksekliğinde olduğu. Hipotez: harf ~1 patch'ten küçükken çözünürlük işe yarar, büyükken yaramaz ya da zarar verir.

In [ ]:
import matplotlib.pyplot as plt
for p in paths:
    img = Image.open(p).convert("RGB"); W, H = img.size
    plt.figure(figsize=(7, 7*H/W)); plt.imshow(img, extent=(0, W, H, 0))
    plt.title(f"{os.path.basename(p)}  ({W}x{H})"); plt.grid(alpha=.35); plt.show()

In [ ]:
# Elle doldur: dosya adı → hedef satırdaki büyük harfin yüksekliği, ORİJİNAL piksel
HARF_PX = {
    # "fis_01.jpg": 38,
    # "kutu_01.jpg": 22,
}
print("girilen:", len(HARF_PX), "/", len(paths))

## Hedef metinler + tam metinler + otomatik puan
- `HEDEF`: puanlamada sayılan ilk sözcükler (fişte işletme adı, kutu/tabelada uyarının başı).
- `TAM_METIN`: fotoğraftaki tüm okunur yazı. Uydurma sütunu bununla hesaplanır. **Bunları fotoğraflara bakarak elle kontrol et**; küçük yazılar ve arkadan görünen ters yazı dahil değil.
- v2 düzeltmesi: fis_01 fotoğrafta `GURME` (ü yok), fis_05 `STI` (Türkçe harfsiz). Tabelada kanun no `4207`.


In [ ]:
import unicodedata, difflib
HEDEF = {
    "kutu_01.jpg":   "sigara dumanında benzen nitrozamin formaldehit",
    "kutu_02.jpg":   "sigara içmek öldürür hemen bırakın",
    "kutu_03.jpg":   "tütün dumanı kansere yol açtığı bilinen 70",
    "tabela_01.jpg": "sigara içilmez burada tütün ürünlerinin",
    "fis_01.jpg": "ENVER BABA GURME GIDA",
    "fis_02.jpg": "eva dondurma",
    "fis_03.jpg": "İBB İŞLETMELER MÜDÜRLÜĞÜ",
    "fis_04.jpg": "KEOBS GIDA LTD ŞTİ",
    "fis_05.jpg": "ANADOLU RESTORAN ISL. LTD. STI.",
    "fis_06.jpg": "GREEN SALADS",
    "fis_07.jpg": "STARBUCKS COFFEE",
}
TAM_METIN = {
    "kutu_01.jpg": "Sigara dumanında benzen, nitrozamin, formaldehit ve hidrojensiyanit gibi kanser yapıcı maddeler bulunur. ALO 171 Sigara Bırakma Danışma Hattı Marlboro Touch Blue",
    "kutu_02.jpg": "Sigara içmek öldürür - hemen bırakın Philip Morris Türkiye SİGARA www.buis.gov.tr SMS 2467",
    "kutu_03.jpg": "Tütün dumanı, kansere yol açtığı bilinen 70'ten fazla madde içerir 8693 5241 Tarım ve Orman Bakanlığı izniyle üretilmiştir",
    "tabela_01.jpg": "SİGARA İÇİLMEZ Burada tütün ürünleri tüketilmesi yasaktır. Tüketenlere ve tüketilmesine göz yuman sorumlulara 4207 sayılı Kanun uyarınca idari para cezası uygulanır. Şikayet için: CEZASI:",
    "fis_01.jpg": "ENVER BABA GURME GIDA SAN.TIC.LTD.ŞTİ. SARIGÖL MAH. CUMHURİYET MEYDANI 16 -17B GAZİOSMANPAŞA İSTANBUL G.O.PAŞA VD 3361143217 TARİH : 01/09/2026 SAAT : 20:26:57 FİŞ NO : 0163 GJ's ICED AMERIC %10 *180,00 TOPKDV *16,36 TOPLAM *180,00 KREDİ *180,00 AKBANK",
    "fis_02.jpg": "EVA DONDURMA MERYEM SARIALİ YILDIZTABYA MAH.YILDIZTABYA CD NO:58/B G.O.P/İSTANBUL G.O.P VD.27173079602 TEŞEKKÜRLER TEŞEKKÜRLER TARİH: 30-08-2026 SAAT: 12:37 FİŞ NO: 0002 DONDURMA %10 *160,00 TOPKDV *14,55 TOPLAM *160,00 Kredi Kartı *160,00",
    "fis_03.jpg": "İBB İŞLETMELER MÜDÜRLÜĞÜ ALTINBOTNUZ SOSYAL TESİSİ MERKEZ MAH. BAHARİYE CAD. HALİÇ TEKKE PARKI İÇİ 16/7 EYÜPSULTAN/İSTANBUL HOCAPAŞA VD. 4810024402 TARİH : 28/08/2026 SAAT : 15:41:56 FİŞ NO : 0086 YİYECEK-İÇECEK %10 *1.765,00 TOPKDV *160,45 TOPLAM *1.765,00 *1.765,00 KATBANK SİCİL NO: EKÜ NO:0001 Z NO:0175 İŞYERİ NO:000000001875800 POS NO:02846735 İŞLEM NO:0092/KP0157 15:42 SATIŞ **** **** **** TUTAR 1.765,00 TL AID : A0000000031010 Visa Credit ONAY KODU : 623187 TUTAR KARŞILIĞI MAL VEYA HİZMETİ ALDIM BU BELGEYİ SAKLAYINIZ Visa payWave ***SADECE TEMASSIZ*** İSTANBUL BÜYÜKŞEHİR BELEDİYESİ SILAHTARAĞA CD. NO: 15 İBB ALTINBOYNUZ SOSYAL TESİSİ 34050 EYÜPSULTAN 34050/İSTANBUL",
    "fis_04.jpg": "KEOBS GIDA LTD ŞTİ ALİBEYKÖY MH 4. GÜL SK NO:9/1 EYÜPSULTAN/İSTANBUL 5441396487 GAZİOSMANPAŞA VD TARİH : 28/08/2026 SAAT : 23:07:40 FİŞ NO : 0075 2,0000Adt X *130,00 FİLTRE KAHVE %10 *260,00 TOPKDV *23,64 TOPLAM *260,00 KREDİ *260,00 VAKIFBANK ************",
    "fis_05.jpg": "ANADOLU RESTORAN ISL.LTD.STI. McD TAKSIM MEYDAN GUMUSSUYU MAH.OSMANLI SOK. TAKSIM BEYOGLU ISTANBUL B.MUKELLEFLER V.D.BSK 6130398713 MERSIS:0613039871300315 TARİH:27/08/2026 SAAT:13:40 FİŞ NO :0046 2 li Kofte Burger M %10 *395,00 (Kofte Burger) (Orta Cola Zero) (Orta Patates) (Ketcap) (Mayonez) (AddOn Istemiyorum) ARA TOPLAM *395,00 TUTAR İND. *-100,00 (%25 indirim) TOPKDV *26,82 TOPLAM *295,00 Kredi Karti Odeme *295,00 KDV KDV TUTARI KDV'Lİ TOPLAM",
    "fis_06.jpg": "GREEN SALADS SADEYEŞİL GIDA SAN.VE TİC. LTD. ŞTİ. YEŞİLPINAR MAH.ŞEHİT METİN KAYA SOK VIALAND AVM NO:11/37 EYÜP/İSTANBUL BAKIRKÖY V.D. 7360645850 TARİH : 23/08/2026 SAAT : 20:46:26 FİŞ NO : 0036 YİYECEK %10 *615,00 TOPKDV *55,91 TOPLAM *615,00 KREDİ İŞ BANKASI *********** *615,00 MERSIS : 073606458500015",
    "fis_07.jpg": "STARBUCKS COFFEE SHAYA KAHVE SAN.VE TİC.A.Ş. Y.PINAR MH. SEHIT METIN KAYA SK.NO:11/172 VIALAND AVM 7690310116 BOĞAZİÇİ KURUMLAR VD. TARİH : 23/08/2026 SAAT : 21:23:25 FİŞ NO : 0261 Cappuccino Grande %10 *225,00 TOPKDV *20,45 TOPLAM *225,00 KREDİ *225,00 YKB ************ TRRG02020853/Çek:3668/Checkseq:4281283713668 REF.NO: 9972222336 YKB (225.00 TL) MERSIS 0769031011600554 www.shayakahve.com.tr İLETİŞİM HATTI : 444 47 49",
}
def tr_lower(s):
    s = unicodedata.normalize("NFC", s).replace("İ", "i").replace("I", "ı")
    return s.lower()
def kelimeler(s): return re.findall(r"[^\W_]+", tr_lower(s))
def harfli(s): return re.findall(r"[^\W_\d]+", tr_lower(s))   # yalnız harfli sözcükler (sayılar uydurma sayılmaz)

def puan(cikti, hedef):
    h = kelimeler(hedef)
    return sum(w in set(kelimeler(cikti)) for w in h) / len(h)      # hedef sözcüklerin kaçı çıktıda aynen var

def uydurma(cikti, tam):
    """Fotoğrafta olmayan VE fotoğraftaki hiçbir sözcüğe benzemeyen (benzerlik < 0.7, yani harf hatası değil) sözcükler (>= 3 harf)."""
    tw = set(harfli(tam)); out = []
    for w in harfli(cikti):
        if len(w) < 3 or w in tw: continue
        if max(difflib.SequenceMatcher(None, w, x).ratio() for x in tw) >= 0.7: continue
        out.append(w)
    return out

df = pd.read_csv("sonuclar_v3.csv")
df["eslesme"] = [puan(str(c), HEDEF[f]) for c, f in zip(df.cikti, df.foto)]
u = [uydurma(str(c), TAM_METIN[f]) for c, f in zip(df.cikti, df.foto)]
df["uydurma_n"] = [len(x) for x in u]; df["uydurma_sozcuk"] = [" ".join(x) for x in u]
df["grup"] = df.foto.str.startswith("fis").map({True: "fiş başlığı", False: "kutu+tabela"})
df.to_csv("sonuclar_v3_puanli.csv", index=False)
print("puanlanan:", df.eslesme.notna().sum(), "/", len(df))


In [9]:
pd.options.display.float_format = "{:.0%}".format
d = df.dropna(subset=["eslesme"]); cols = list(PROMPTS)

# Önce her koşu kendi içinde ortalanır, sonra koşular arası min / ort / maks alınır.
per_run = d.groupby(["kosu","grup","prompt","cozunurluk"]).eslesme.mean().reset_index()
per_run_all = d.groupby(["kosu","prompt","cozunurluk"]).eslesme.mean().reset_index().assign(grup="hepsi")
pr = pd.concat([per_run, per_run_all])
agg = pr.groupby(["grup","prompt","cozunurluk"]).eslesme.agg(["min","mean","max"]).unstack("cozunurluk")
print("== Tablo 1: ortalama sözcük eşleşmesi, koşular arası min / ort / maks (224 | 448) ==")
for g in ["kutu+tabela","fiş başlığı","hepsi"]:
    print("\n", g); print(agg.loc[g].reindex(cols))

print("\n== Fiş başlıkları: her koşu ayrı (224 | 448), ocr ==")
print(per_run[(per_run.grup=="fiş başlığı")&(per_run.prompt=="ocr")].pivot(index="kosu", columns="cozunurluk", values="eslesme"))

print("\n== Tablo 2: fotoğraf başına, ocr, koşular arası min–maks ==")
t2 = d[d.prompt=="ocr"].groupby(["foto","cozunurluk"]).eslesme.agg(["min","max"]).unstack("cozunurluk"); print(t2)

print("\n== Hangi çıktılar koşudan koşuya değişti? (aynı foto × çözünürlük × prompt, farklı metin sayısı) ==")
ch = d.groupby(["foto","cozunurluk","prompt"]).cikti.nunique()
print((ch > 1).groupby(level="prompt").sum(), "/", len(ch), "hücre")

print("\n== Uydurma: fotoğrafta olmayan sözcük sayısı (ocr, koşu ortalaması) ==")
pd.options.display.float_format = "{:.1f}".format
print(d[d.prompt=="ocr"].pivot_table(index="foto", columns="cozunurluk", values="uydurma_n", aggfunc="mean"))
print("\nGrup toplamı (koşu başına, ocr):")
print(d[d.prompt=="ocr"].groupby(["grup","cozunurluk","kosu"]).uydurma_n.sum().unstack("kosu"))
print("\nÖrnek uydurma sözcükler (ocr, 224):")
print(d[(d.prompt=="ocr")&(d.cozunurluk==224)&(d.kosu==1)][["foto","uydurma_sozcuk"]].to_string(index=False))


== Tablo 1: ortalama sözcük eşleşmesi, koşular arası min / ort / maks (224 | 448) ==

 kutu+tabela
           min     mean     max    
cozunurluk 224 448  224 448 224 448
prompt                             
ocr        44% 86%  44% 86% 44% 86%
en_en      44% 86%  44% 86% 44% 86%
tr_tr      35% 81%  35% 81% 35% 81%
tr_en      44% 86%  44% 86% 44% 86%
en_tr      20% 56%  20% 56% 20% 56%

 fiş başlığı
           min     mean     max    
cozunurluk 224 448  224 448 224 448
prompt                             
ocr        84% 78%  84% 78% 84% 78%
en_en      80% 78%  80% 78% 80% 78%
tr_tr      29% 39%  29% 39% 29% 39%
tr_en      80% 71%  80% 71% 80% 71%
en_tr      21% 39%  21% 39% 21% 39%

 hepsi
           min     mean     max    
cozunurluk 224 448  224 448 224 448
prompt                             
ocr        70% 81%  70% 81% 70% 81%
en_en      67% 81%  67% 81% 67% 81%
tr_tr      31% 55%  31% 55% 31% 55%
tr_en      67% 77%  67% 77% 67% 77%
en_tr      21% 46%  21% 46% 21% 46%

== Fiş başlıkl

In [ ]:
# Harf yüksekliği ↔ 224→448 farkı (ocr). HARF_PX yukarıda doldurulduysa çalışır.
if HARF_PX:
    o = d[d.prompt=="ocr"].pivot_table(index="foto", columns="cozunurluk", values="eslesme", aggfunc="mean")
    o.columns = ["ocr_224","ocr_448"]
    wh = df.drop_duplicates("foto").set_index("foto")[["W","H"]]
    o = o.join(wh)
    o["harf_px"] = pd.Series(HARF_PX)
    o = o.dropna(subset=["harf_px"])
    o["harf_patch_224"] = o.harf_px * 224 / o.H / 14
    o["harf_patch_448"] = o.harf_px * 448 / o.H / 14
    o["fark_448_224"] = o.ocr_448 - o.ocr_224
    pd.options.display.float_format = "{:.2f}".format
    print(o[["harf_px","harf_patch_224","harf_patch_448","ocr_224","ocr_448","fark_448_224"]].sort_values("harf_patch_224"))
else:
    print("Önce HARF_PX'i doldur.")

## Yazıya taşırken
- Oturumlar arası min–maks için `compare_sessions.py`. Bu notebook'taki "koşular arası" tablolar aynı oturumun tekrarı olduğundan min = ort = maks çıkar.
- Fiş başlıkları `ocr` 224 oturumlar arasında 70–84, 448 sabit 78: "kazanç yok" doğru, "448 daha kötü" denemez.
- Çapraz hücreler: `tr_en` ve `en_tr`. `tr_en` düşükse etken **dil kodu**, `en_tr` düşükse etken **sorunun dili**.
- Sınırlar: N=11, greedy ama oturumlar arası oynama var, uydurma ölçütü `TAM_METIN` okumasına bağlı.
